# Data Quality — Rerun Specific Check Notebook

**Use this notebook to rerun a specific check on demand, regardless of `run_frequency` or whether it already ran today.**

**Steps:**
1. Run cell 1 — set your Lakehouse/Schema names
2. Run cell 2 — see all available checks
3. Run cell 3 — enter which check (and optionally which model) to rerun
4. Run cell 4 — executes the check and writes new results
5. Run cell 5 — view results

New results are written with a fresh `run_id` — previous results are preserved.

In [ ]:
# -- Step 1: Configuration -----------------------------------------------------
# Load shared settings from config.py in Lakehouse.
from datetime import datetime, timezone
import uuid
import sys
sys.path.append("/lakehouse/default/Files/code")

try:
    from config import LAKEHOUSE_NAME, SCHEMA_NAME, MAX_RETRY_ATTEMPTS, INITIAL_RETRY_DELAY, DAX_TIMEOUT_SECONDS, RUN_TABLE_MAINTENANCE, MAINTENANCE_DAY_UTC
except ImportError:
    # Fallback keeps the notebook runnable if shared config execution is unavailable.
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"
    MAX_RETRY_ATTEMPTS = 3
    INITIAL_RETRY_DELAY = 2
    DAX_TIMEOUT_SECONDS = 60
    RUN_TABLE_MAINTENANCE = False
    MAINTENANCE_DAY_UTC = 6

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"
print(f"Schema: {FULL_SCHEMA}")

In [ ]:
# -- Step 2: List all available checks -----------------------------------------
from IPython.display import display

all_checks = spark.sql(f"""
    SELECT
        r.check_name,
        r.model_name,
        r.workspace_id,
        r.dataset_id,
        r.workspace_name,
        r.dataset_name,
        r.run_frequency,
        r.fail_delta_pct_threshold,
        r.is_baseline,
        COALESCE(c.baseline_mode, 'MODEL') AS baseline_mode,
        c.static_baseline_value
    FROM {FULL_SCHEMA}.check_registry r
    LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
      ON r.check_name = c.check_name
    WHERE r.is_active = true
    ORDER BY r.check_name, r.model_name
""").toPandas()

if len(all_checks) == 0:
    print("No active checks found. Add checks using 02_data_quality_manage_checks_notebook first.")
else:
    print(f"Available checks ({len(all_checks)} active rows):\n")
    display(all_checks[[
        "check_name", "model_name", "workspace_id", "dataset_id", "workspace_name", "dataset_name",
        "run_frequency", "fail_delta_pct_threshold", "is_baseline", "baseline_mode", "static_baseline_value"
    ]])

In [ ]:
# ── Step 3: Enter which check to rerun ──────────────────────────────
# Use values from the table above.
# Identity-safe target selection uses model_name + dataset_name and resolves IDs internally.

CHECK_NAME = ""  # required: e.g. "My Check 1"
MODEL_NAME = None
DATASET_NAME = None
WORKSPACE_NAME = None

if MODEL_NAME and not DATASET_NAME:
    raise ValueError("dataset_name is required when model_name is provided.")

print(f"\nSelected check:     {CHECK_NAME}")
print(f"Selected model:     {MODEL_NAME if MODEL_NAME else '(all models)'}")
print(f"Selected dataset:   {DATASET_NAME if DATASET_NAME else '(all datasets)'}")
print(f"Selected workspace: {WORKSPACE_NAME if WORKSPACE_NAME else '(all workspaces)'}")

In [ ]:
# -- Step 4: Execute and write results -----------------------------------------
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, TimestampType

# Validate sempy is available
try:
    import sempy.fabric as sempy
except ImportError:
    raise RuntimeError(
        "sempy library not available. This notebook must run in Microsoft Fabric. "
        "Visit: https://learn.microsoft.com/en-us/python/api/sempy/"
    )

now_utc = datetime.now(timezone.utc)
RUN_DATE = now_utc.date()
RUN_TIMESTAMP = now_utc.replace(tzinfo=None)
RUN_ID = str(uuid.uuid4())

if not CHECK_NAME:
    raise ValueError("check_name is required. Re-run step 3 and provide a value.")

# Escape user inputs to keep dynamic SQL safe
CHECK_NAME_SQL = CHECK_NAME.replace("'", "''")
MODEL_NAME_SQL = MODEL_NAME.replace("'", "''") if MODEL_NAME else None
DATASET_NAME_SQL = DATASET_NAME.replace("'", "''") if DATASET_NAME else None
WORKSPACE_NAME_SQL = WORKSPACE_NAME.replace("'", "''") if WORKSPACE_NAME else None

# Build optional filters.
model_filter = f"AND model_name = '{MODEL_NAME_SQL}'" if MODEL_NAME_SQL else ""
dataset_filter = f"AND dataset_name = '{DATASET_NAME_SQL}'" if DATASET_NAME_SQL else ""
workspace_filter = f"AND workspace_name = '{WORKSPACE_NAME_SQL}'" if WORKSPACE_NAME_SQL else ""

# Validate selection and resolve stable IDs
checks_to_run = spark.sql(f"""
    SELECT check_name, model_name, workspace_id, dataset_id, workspace_name, dataset_name, dax_expression, fail_delta_pct_threshold
    FROM {FULL_SCHEMA}.check_registry
    WHERE check_name = '{CHECK_NAME_SQL}'
      AND is_active = true
      {model_filter}
      {dataset_filter}
      {workspace_filter}
    ORDER BY model_name, dataset_name, workspace_name
""").toPandas()

if len(checks_to_run) == 0:
    raise ValueError(
        f"No active checks found for check_name='{CHECK_NAME}'"
        + (f", model_name='{MODEL_NAME}'" if MODEL_NAME else "")
        + (f", dataset_name='{DATASET_NAME}'" if DATASET_NAME else "")
        + (f", workspace_name='{WORKSPACE_NAME}'" if WORKSPACE_NAME else "")
        + ". Check the values above and re-run step 3."
    )

if MODEL_NAME and DATASET_NAME and len(checks_to_run) > 1 and not WORKSPACE_NAME:
    preview = checks_to_run[["model_name", "dataset_name", "workspace_name", "workspace_id", "dataset_id"]]
    raise ValueError(
        "Selection matched multiple active identity rows. Provide workspace_name in step 3 to disambiguate.\n"
        + preview.to_string(index=False)
    )

missing_ids = checks_to_run[
    checks_to_run["workspace_id"].isna()
    | checks_to_run["dataset_id"].isna()
    | (checks_to_run["workspace_id"].astype(str).str.strip() == "")
    | (checks_to_run["dataset_id"].astype(str).str.strip() == "")
]
if len(missing_ids) > 0:
    raise RuntimeError("workspace_id/dataset_id are required for rerun rows. Update check_registry first.")

invalid_thresholds = checks_to_run[
    checks_to_run["fail_delta_pct_threshold"].isna()
    | (checks_to_run["fail_delta_pct_threshold"] < 0)
]
if len(invalid_thresholds) > 0:
    raise RuntimeError(
        "Rerun target rows must have non-negative fail_delta_pct_threshold values.\n"
        + invalid_thresholds[["check_name", "model_name", "workspace_id", "dataset_id", "fail_delta_pct_threshold"]].to_string(index=False)
    )

print(f"Run ID: {RUN_ID}")
print(f"Rerunning {len(checks_to_run)} check(s) for '{CHECK_NAME}'...\n")
print(f"DAX timeout: {DAX_TIMEOUT_SECONDS}s")
print("Resolved identity rows:")
print(checks_to_run[["model_name", "dataset_name", "workspace_name", "workspace_id", "dataset_id", "fail_delta_pct_threshold"]].to_string(index=False))

RESULT_BATCH_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("run_date", DateType(), True),
    StructField("run_timestamp", TimestampType(), True),
    StructField("check_name", StringType(), True),
    StructField("model_name", StringType(), True),
    StructField("workspace_id", StringType(), True),
    StructField("dataset_id", StringType(), True),
    StructField("workspace_name", StringType(), True),
    StructField("dataset_name", StringType(), True),
    StructField("result_value", DoubleType(), True),
    StructField("baseline_model", StringType(), True),
    StructField("baseline_value", DoubleType(), True),
    StructField("absolute_delta", DoubleType(), True),
    StructField("delta_pct", DoubleType(), True),
    StructField("fail_delta_pct_threshold", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
])

# Helper: Call sempy.evaluate_dax across sempy versions.
def evaluate_dax_compat(dataset, workspace, dax):
    try:
        return sempy.evaluate_dax(dataset=dataset, workspace=workspace, dax_string=dax)
    except TypeError as exc:
        if "unexpected keyword argument 'dax_string'" not in str(exc):
            raise
        return sempy.evaluate_dax(dataset=dataset, workspace=workspace, dax=dax)

# Helper: Execute a single DAX with timeout protection
def evaluate_dax_with_timeout(dataset, workspace, dax, timeout_seconds=DAX_TIMEOUT_SECONDS):
    executor = ThreadPoolExecutor(max_workers=1)
    future = executor.submit(evaluate_dax_compat, dataset, workspace, dax)
    try:
        return future.result(timeout=timeout_seconds), None
    except FuturesTimeoutError:
        future.cancel()
        return None, f"DAX execution timed out after {timeout_seconds}s"
    except Exception as e:
        return None, str(e)
    finally:
        executor.shutdown(wait=False, cancel_futures=True)

# Helper: Retry with exponential backoff
def evaluate_dax_with_retry(dataset, workspace, dax):
    for attempt in range(1, MAX_RETRY_ATTEMPTS + 1):
        result_df, error_msg = evaluate_dax_with_timeout(dataset, workspace, dax)
        if error_msg is None:
            return result_df, None
        if attempt < MAX_RETRY_ATTEMPTS:
            delay = INITIAL_RETRY_DELAY ** attempt
            print(f"      Retry {attempt}/{MAX_RETRY_ATTEMPTS} failed, retrying in {delay}s...")
            time.sleep(delay)
        else:
            return None, error_msg
    return None, "Max retries exceeded"

# Helper: Extract numeric value with type validation
def extract_numeric_value(result_df):
    if result_df is None or len(result_df) == 0:
        return None, "DAX returned empty result"
    try:
        val = result_df.iloc[0, 0]
        if isinstance(val, (int, float)):
            return float(val), None
        if isinstance(val, str):
            try:
                return float(val), None
            except ValueError:
                return None, f"DAX returned non-numeric string: {val}"
        if val is None:
            return None, "DAX returned NULL"
        return None, f"DAX returned unsupported type: {type(val).__name__}"
    except Exception as e:
        return None, f"Failed to extract value: {str(e)[:200]}"

# Helper: MERGE batch for idempotent writes
def merge_results_batch(batch_rows):
    if not batch_rows:
        return

    batch_df = spark.createDataFrame(batch_rows, schema=RESULT_BATCH_SCHEMA)
    batch_df.createOrReplaceTempView("dq_rerun_check_batch")

    spark.sql(f"""
    MERGE INTO {FULL_SCHEMA}.validation_results t
    USING dq_rerun_check_batch s
      ON t.run_id = s.run_id
     AND t.check_name = s.check_name
     AND t.workspace_id = s.workspace_id
     AND t.dataset_id = s.dataset_id
    WHEN MATCHED THEN UPDATE SET
      t.run_date = s.run_date,
      t.run_timestamp = s.run_timestamp,
      t.model_name = s.model_name,
      t.workspace_id = s.workspace_id,
      t.dataset_id = s.dataset_id,
      t.workspace_name = s.workspace_name,
      t.dataset_name = s.dataset_name,
      t.result_value = s.result_value,
      t.baseline_model = s.baseline_model,
      t.baseline_value = s.baseline_value,
      t.absolute_delta = s.absolute_delta,
      t.delta_pct = s.delta_pct,
      t.fail_delta_pct_threshold = s.fail_delta_pct_threshold,
      t.status = s.status,
      t.error_message = s.error_message
    WHEN NOT MATCHED THEN INSERT (
      run_id, run_date, run_timestamp, check_name, model_name,
      workspace_id, dataset_id, workspace_name, dataset_name,
      result_value, baseline_model, baseline_value, absolute_delta,
      delta_pct, fail_delta_pct_threshold, status, error_message
    ) VALUES (
      s.run_id, s.run_date, s.run_timestamp, s.check_name, s.model_name,
      s.workspace_id, s.dataset_id, s.workspace_name, s.dataset_name,
      s.result_value, s.baseline_model, s.baseline_value, s.absolute_delta,
      s.delta_pct, s.fail_delta_pct_threshold, s.status, s.error_message
    )
    """)

# Preflight target connectivity
probe_targets = checks_to_run[["workspace_id", "dataset_id", "workspace_name", "dataset_name"]].drop_duplicates()
for _, target in probe_targets.iterrows():
    _, probe_err = evaluate_dax_with_retry(
        target["dataset_id"],
        target["workspace_id"],
        'EVALUATE ROW("probe", 1)'
    )
    if probe_err:
        print(f"  Connectivity warning for {target['workspace_name']} / {target['dataset_name']}: {probe_err}")

# Resolve baseline mode for this check
baseline_cfg = spark.sql(f"""
    SELECT COALESCE(baseline_mode, 'MODEL') AS baseline_mode, static_baseline_value
    FROM {FULL_SCHEMA}.check_baseline_config
    WHERE check_name = '{CHECK_NAME_SQL}'
""").toPandas()

if len(baseline_cfg) == 0:
    baseline_mode = "MODEL"
    static_baseline_value = None
else:
    baseline_mode = str(baseline_cfg.iloc[0]["baseline_mode"]).strip().upper()
    static_baseline_value = baseline_cfg.iloc[0]["static_baseline_value"]

if baseline_mode not in ["MODEL", "STATIC"]:
    raise RuntimeError(f"Invalid baseline_mode '{baseline_mode}' for check_name={CHECK_NAME}")

baseline_model = None
baseline_value = None

if baseline_mode == "STATIC":
    if static_baseline_value is None:
        raise RuntimeError(
            f"STATIC baseline_mode requires static_baseline_value for check_name={CHECK_NAME}. "
            "Set it via 02_data_quality_manage_checks_notebook."
        )
    baseline_model = "STATIC_VALUE"
    baseline_value = float(static_baseline_value)
    print(f"Baseline (STATIC): {baseline_value}\n")
else:
    baseline_rows = spark.sql(f"""
        SELECT model_name, workspace_id, dataset_id, workspace_name, dataset_name, dax_expression
        FROM {FULL_SCHEMA}.check_registry
        WHERE check_name = '{CHECK_NAME_SQL}'
          AND is_active = true
          AND is_baseline = true
    """).toPandas()
    if len(baseline_rows) != 1:
        raise RuntimeError(
            f"Baseline invariant violation for check_name={CHECK_NAME}: expected exactly 1 active baseline row, found {len(baseline_rows)}. "
            "Fix baseline assignment via 02_data_quality_manage_checks_notebook."
        )
    baseline_row = baseline_rows.iloc[0]
    baseline_model = baseline_row["model_name"]
    b_result_df, b_error = evaluate_dax_with_retry(
        baseline_row["dataset_id"],
        baseline_row["workspace_id"],
        baseline_row["dax_expression"]
    )
    if b_error:
        print(f"Baseline ({baseline_model}): ERROR - {b_error}\n")
    else:
        baseline_value, extract_error = extract_numeric_value(b_result_df)
        if extract_error:
            print(f"Baseline ({baseline_model}): TYPE ERROR - {extract_error}\n")
        else:
            print(f"Baseline ({baseline_model}): {baseline_value}\n")

pass_count = fail_count = error_count = 0
results_batch = []
num_rows = len(checks_to_run)

for row_idx, (_, row) in enumerate(checks_to_run.iterrows()):
    model_name = row["model_name"]
    row_threshold = float(row["fail_delta_pct_threshold"])
    result_value = absolute_delta = delta_pct = error_msg = None
    status = "PASS"

    result_df, exec_error = evaluate_dax_with_retry(
        row["dataset_id"],
        row["workspace_id"],
        row["dax_expression"]
    )

    if exec_error:
        status = "ERROR"
        error_msg = exec_error[:500]
        error_count += 1
    else:
        value, extract_error = extract_numeric_value(result_df)
        if extract_error:
            status = "ERROR"
            error_msg = extract_error
            error_count += 1
        else:
            result_value = value
            if baseline_value is not None and result_value is not None:
                absolute_delta = result_value - baseline_value
                if baseline_value != 0:
                    delta_pct = (absolute_delta / baseline_value) * 100
                    status = "FAIL" if abs(delta_pct) > row_threshold else "PASS"
                else:
                    status = "FAIL" if result_value != 0 else "PASS"
            elif baseline_value is None:
                status = "ERROR"
                error_msg = "Baseline is unavailable"
                error_count += 1
            else:
                status = "ERROR"
                error_msg = "Result value is None despite no exception"
                error_count += 1

            if status == "PASS":
                pass_count += 1
            elif status == "FAIL":
                fail_count += 1

    results_batch.append({
        "run_id": RUN_ID,
        "run_date": RUN_DATE,
        "run_timestamp": RUN_TIMESTAMP,
        "check_name": CHECK_NAME,
        "model_name": model_name,
        "workspace_id": row["workspace_id"],
        "dataset_id": row["dataset_id"],
        "workspace_name": row["workspace_name"],
        "dataset_name": row["dataset_name"],
        "result_value": result_value,
        "baseline_model": baseline_model,
        "baseline_value": baseline_value,
        "absolute_delta": absolute_delta,
        "delta_pct": delta_pct,
        "fail_delta_pct_threshold": row_threshold,
        "status": status,
        "error_message": error_msg
    })

    if len(results_batch) >= 100 or row_idx == num_rows - 1:
        merge_results_batch(results_batch)
        results_batch = []

print("\nRerun complete")
print(f"    PASS:  {pass_count}")
print(f"    FAIL:  {fail_count}")
print(f"    ERROR: {error_count}")

## Results

In [ ]:
from IPython.display import display

if 'RUN_ID' not in locals():
    print("⚠️  RUN_ID not found. Please run the prior cells to execute a rerun first.")
else:
    results = spark.sql(f"""
        SELECT check_name, model_name, result_value, baseline_value, delta_pct, fail_delta_pct_threshold, status, error_message
        FROM {FULL_SCHEMA}.validation_results
        WHERE run_id = '{RUN_ID}'
        ORDER BY model_name
    """).toPandas()
    
    display(results)